# Import Libraries

In [1]:
!apt-get update
!apt-get install -y tshark

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,924 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,821 kB]
Get:14 http://

In [1]:
import os
import csv
import ast
import requests
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.preprocessing import LabelEncoder
pd.set_option('display.max_rows', None)

# Get Packet Fields

In [12]:
def get_nested_field_names(field_element):
    nested_field_names = []
    for nested_field in field_element.findall('field'):
        field_name = nested_field.get('name').replace(".", "_")
        if field_name == '':
         #   continue
         field_name = 'nested_empty_field_name'
        nested_field_names.extend([field_name + '_name', field_name + '_showname', field_name + '_size',
                                              field_name + '_pos', field_name + '_show', field_name + '_value'])
        nested_field_names.extend(get_nested_field_names(nested_field))  # Recursively collect nested fields
    return nested_field_names

In [11]:
def get_packet_field_names(proto):
    packet_fields = []

    for field in proto.findall('field'):
        field_name = field.get('name')
        if not field_name:
            field_name = 'empty_field_name'

        field_name = field_name.replace(".", "_")

        packet_fields.extend([
            field_name + '_name',
            field_name + '_showname',
            field_name + '_size',
            field_name + '_pos',
            field_name + '_show',
            field_name + '_value'
        ])

        packet_fields.extend(get_nested_field_names(field))

    return packet_fields

# Align Packet Features

In [10]:
def align_lists(list1, list2, list2_values):
    list2_values_aligned = []
    list2_aligned = []
    for x in list1:
        if x not in list2:
            list2_values_aligned.append('')
        else:
            idx = list2.index(x)
            list2_values_aligned.append(list2_values[idx])
    return list2_values_aligned

# Make Dataframe from Packet Features

In [9]:
def get_nested_fields(field_element):
    nested_fields = []
    for nested_field in field_element.findall('field'):
        #if nested_field.get('name') == '':
        #    continue
        nested_fields.extend([
                nested_field.get('name'),
                nested_field.get('showname'),
                nested_field.get('size'),
                nested_field.get('pos'),
                nested_field.get('show'),
                nested_field.get('value')
            ])
        nested_fields.extend(get_nested_fields(nested_field))  # Recursively collect nested fields
    return nested_fields

In [8]:
def makeDataframe(xmlfile):
    dataframe_list = []
    tree = ET.parse(xmlfile)
    root = tree.getroot()
    packets = root.findall('packet')

    column_names = []
    values = []
    flag = False

    # -------- First pass: collect column names --------
    for packet in packets:
        for proto in packet.findall('proto'):
            if proto.get('name') == 'nr-rrc':
                if proto.get('hide') == 'yes':
                  continue
                print("TEST")
                print(proto.get('name'))
                columns = get_packet_field_names(proto)
                column_names = sorted(set(column_names).union(set(columns)))
                for nas_proto in proto.iter("proto"):
                    if nas_proto.get("name") == "nas-5gs":
                      print(nas_proto.get('name'))
                      columns = get_packet_field_names(nas_proto)
                      column_names = sorted(set(column_names).union(set(columns)))
    column_names = sorted(column_names)

    # -------- Second pass: extract values --------
    for packet in packets:
        for proto in packet.findall('proto'):
            if proto.get('name') == 'nr-rrc':
                if proto.get('hide') == 'yes':
                  continue
                #flag = True
                packet_fields = []
                columns = []
                print("RRC")
                # extract all fields inside NGAP
                for field in proto.findall('field'):
                  #if not field.get('name'):
                  #    continue
                  packet_fields.extend([
                      field.get('name'),
                      field.get('showname'),
                      field.get('size'),
                      field.get('pos'),
                      field.get('show'),
                      field.get('value')
                  ])
                  packet_fields.extend(get_nested_fields(field))
                columns += get_packet_field_names(proto)

                for nas_proto in proto.iter("proto"):
                  print("NAS")
                  if nas_proto.get("name") == "nas-5gs":
                    for field in nas_proto.findall("field"):
                      packet_fields.extend([
                          field.get('name'),
                          field.get('showname'),
                          field.get('size'),
                          field.get('pos'),
                          field.get('show'),
                          field.get('value')
                      ])
                      packet_fields.extend(get_nested_fields(field))
                    columns += get_packet_field_names(nas_proto)

                values.append(align_lists(column_names, columns, packet_fields))

    print("Column names:", column_names)
    print("Number of rows:", len(values))
    return list(column_names), values

# Prepare Dataframe from PCAP File

In [4]:
import subprocess

def run_tshark_to_pdml(input_file, xml_output_file):
    # Open the output file in write mode
    with open(xml_output_file, "w") as f:
        # Run tshark - read input_file, output PDML, write stdout to f
        subprocess.run(["tshark", "-r", input_file, "-T", "pdml"], stdout=f, check=True)

In [2]:
def prepare_dataframe_from_pcap_file(xml_output_file):
    #xml_output_file = input_file.replace("pcapng", "xml")
    #run_tshark_to_pdml(input_file, xml_output_file)
    try:
      column_names, values = makeDataframe(xml_output_file)
    except:
      print("error")
    df = pd.DataFrame(values, columns=column_names)
    return df

In [31]:
df = pd.DataFrame()
# ...existing code...
xml_path = os.path.join(os.getcwd(), "XML_files", "reject_cause_27.xml")
df_i = prepare_dataframe_from_pcap_file(xml_path)
# ...existing code...
try:
    df_i = prepare_dataframe_from_pcap_file(xml_path)
except IsADirectoryError as e:
    print("!!!error")
print(df_i.empty)
df = pd.concat([df, df_i])
df["label"] = 0

TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
nas-5gs
TEST
nr-rrc
nas-5gs
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
RRC
NAS
NAS
NAS
NAS
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
RRC
NAS
NAS
RRC
NAS
NAS
Column names: ['e212_mcc_name', 'e212_mcc_pos', 'e212_mcc_show', 'e212_mcc_showname', 'e212_mcc_size', 'e212_mcc_value', 'e212_mnc_name', 'e212_mnc_pos', 'e212_mnc_show', 'e212_mnc_showname', 'e212_mnc_size', 'e212_mnc_value', 'empty_field_name_name', 'empty_field_name_pos', 'empty_field_name_show', 'empty_field_name_showname', 'empty_field_name_size', 'empty_field_name_value', 'gsm_a_len_name', 'gsm_a_len_pos', 'gsm_a_len_show', 'gsm_a_len_showname', 'gsm_a_len_size', 'gsm_a_len_value', 'nas-5gs_epd_name', 'nas-5gs_epd_pos', 'nas-5gs_epd_show', 'nas-5gs_epd_showname', 'nas-5gs_epd_size', 'nas-5gs_epd_value', 'nas-5gs_mm_128_5g_ea1_name', 'nas-5gs_mm_128_5g_ea1_pos', 'nas-5gs_mm_128_5g_ea1_show', 'nas-5gs_mm_128_5g_ea1_showname', 'nas-5gs_mm_128_5g_ea1_size'

In [28]:
#os.makedirs("../output", exist_ok=True)
xml_path_csv = os.path.join(os.getcwd(), "csv_files_in_order_raw", "reject_cause_27_raw.csv")
df.to_csv(xml_path_csv, index=False)

## Encode Categorical Values

In [29]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
df[categorical_cols] = df[categorical_cols].fillna("Unknown")
le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

In [32]:
df = df.replace(r'^\s*$', np.nan, regex=True)

# Encode categorical columns, preserving missing as -1
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

for col in categorical_cols:
    codes, _ = pd.factorize(df[col], use_na_sentinel=True)  # NaN -> -1
    df[col] = codes.astype(int)

# Fill remaining numeric NaN with -1
df = df.fillna(-1)

C:\Users\Asus\AppData\Local\Temp\ipykernel_20024\1892130464.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(r'^\s*$', np.nan, regex=True)


In [34]:
xml_path_csv = os.path.join(os.getcwd(), "csv_files", "reject_cause_27_fill_1.csv")
df.to_csv(xml_path_csv, index=False)

## Handle Missing values

In [24]:
xml_path_csv = os.path.join(os.getcwd(), "csv_files_filtered", "normal_traffic_neg1.csv")
df.to_csv(xml_path_csv, index=False)

In [35]:
xml_folder = os.path.join(os.getcwd(), "XML_files")
xml_files = sorted([f for f in os.listdir(xml_folder) if f.lower().endswith(".xml")])

all_frames = []
session_map = {}  # session_name -> Sequence_Number

for idx, file_name in enumerate(xml_files, start=1):
    xml_path = os.path.join(xml_folder, file_name)
    session_name = os.path.splitext(file_name)[0]   # e.g., reject_cause_27
    session_map[session_name] = idx

    try:
        df_i = prepare_dataframe_from_pcap_file(xml_path)
    except Exception as e:
        print(f"Skipping {file_name}: {e}")
        continue

    if df_i is None or df_i.empty:
        print(f"Skipping {file_name}: empty dataframe")
        continue

    # keep only sequence id per session
    df_i["Sequence_Number"] = idx
    all_frames.append(df_i)

if not all_frames:
    raise ValueError(f"No valid XML dataframes produced from folder: {xml_folder}")

# Outer join alignment across all packet feature columns
df = pd.concat(all_frames, axis=0, join="outer", ignore_index=True, sort=False)

# Normalize blanks, then fill missing features with 0.0
df = df.replace(r'^\s*$', np.nan, regex=True)
df = df.fillna(-1)

# Optional: keep Sequence_Number as integer
df["Sequence_Number"] = df["Sequence_Number"].astype(int)

# Save combined CSV
out_csv = os.path.join(os.getcwd(), "csv_files", "all_sessions_aligned.csv")
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
df.to_csv(out_csv, index=False)

print(f"Saved: {out_csv}")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
print("Session mapping:", session_map)

TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
TEST
nr-rrc
nas-5gs
TEST
nr-rrc
nas-5gs
TEST
nr-rrc
nas-5gs
TEST
nr-rrc
nas-5gs
TEST
nr-rrc
TEST
nr-rrc
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
RRC
NAS
NAS
NAS
NAS
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
NAS
RRC
NAS
NAS
Column names: ['e212_mcc_name', 'e212_mcc_pos', 'e212_mcc_show', 'e212_mcc_showname', 'e212_mcc_size', 'e212_mcc_value', 'e212_mnc_name', 'e212_mnc_pos', 'e212_mnc_show', 'e212_mnc_showname', 'e212_mnc_size', 'e212_mnc_value', 'empty_field_name_name', 'empty_field_name_pos', 'empty_field_name_show', 'empty_field_name_showname', 'empty_field_name_size', 'empty_field_name_value', 'gsm_a_dtap_autn_amf_name', 'gsm_a_dtap_autn_amf_pos', 'gsm_a_dtap_autn_amf_show', 'gsm_a_dtap_autn_amf_showname', 'gsm_a_dtap_autn_amf_size', 'gsm_a_dtap_autn_amf_value', 'gsm_a_dtap_autn_mac_name', 'gsm_a_dtap_autn_mac_pos', 'gsm_a_dtap_autn_mac_show', 'gsm_a_dtap_autn_mac_showname', 'gsm_a_dtap_autn_mac_size', 'gsm_a_dtap_autn_mac_value',

C:\Users\Asus\AppData\Local\Temp\ipykernel_20024\495311483.py:33: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(r'^\s*$', np.nan, regex=True)


Saved: c:\thesis_SNC\thesis_SNC\csv_files\all_sessions_aligned.csv
Rows: 121, Columns: 872
Session mapping: {'MAC_failure': 1, 'normal_traffic': 2, 'reject_cause_11': 3, 'reject_cause_13': 4, 'reject_cause_15': 5, 'reject_cause_24': 6, 'reject_cause_27': 7, 'reject_cause_7': 8}


## Label Dataset

In [ ]:
# Rule-based labels:
# normal_traffic -> 0
# all other sessions -> 1
normal_seq = session_map.get("normal_traffic")
if normal_seq is None:
    raise ValueError("Session 'normal_traffic' not found in session_map.")

df["label"] = (df["Sequence_Number"] != normal_seq).astype(int)

# drop Session_Name if present
if "Session_Name" in df.columns:
    df = df.drop(columns=["Session_Name"])

# encode all remaining non-numeric feature columns
protected = {"Sequence_Number", "label"}
feature_cols = [c for c in df.columns if c not in protected]

for col in feature_cols:
    if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
        codes, _ = pd.factorize(df[col], use_na_sentinel=True)  # NaN -> -1
        df[col] = codes.astype(int)
    else:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# fill any remaining NaN in features
df[feature_cols] = df[feature_cols].fillna(0.0)

# put Sequence_Number first, label second
ordered_cols = ["Sequence_Number", "label"] + [c for c in df.columns if c not in ["Sequence_Number", "label"]]
df = df[ordered_cols]

# quick check
print(df[["Sequence_Number", "label"]].head(20))
print(df["label"].value_counts(dropna=False))

ValueError: Length of values (0) does not match length of index (18)

In [ ]:
df['label'].value_counts()

## Save processed dataset

In [57]:
df.to_csv("../content/normal_traffic.csv")